In [8]:
import torch
import time

device = 'cpu'  # usa CPU

# Tensore di prova
x = torch.randn(500, 500, device=device)
window_length = 5

# --- Bitonic / odd-even sort senza padding, step=1 ---
def median_bitonic_step1(x, window_length, dim=-1):
    dim = dim % x.ndim
    if dim != x.ndim - 1:
        x = x.movedim(dim, -1)
    
    # finestre di lunghezza window_length, step=1
    windows = x.unfold(-1, window_length, 1)
    
    sorted_vals = windows.clone()
    n = window_length
    for i in range(n):
        for j in range(i % 2, n - 1, 2):
            a = sorted_vals[..., j]
            b = sorted_vals[..., j + 1]
            sorted_vals[..., j] = torch.min(a, b)
            sorted_vals[..., j + 1] = torch.max(a, b)
    
    median_vals = sorted_vals[..., n // 2]
    
    if dim != x.ndim - 1:
        median_vals = median_vals.movedim(-1, dim)
    
    return median_vals

# --- Eliminazione progressiva senza padding, step=1 ---
def median_elimination_step1(x, window_length, dim=-1):
    dim = dim % x.ndim
    if dim != x.ndim - 1:
        x = x.movedim(dim, -1)
    
    windows = x.unfold(-1, window_length, 1)
    vals = windows.clone()
    n = window_length
    for _ in range(n // 2):
        max_idx = vals.argmax(dim=-1, keepdim=True)
        min_idx = vals.argmin(dim=-1, keepdim=True)
        mask = torch.ones_like(vals, dtype=torch.bool)
        mask.scatter_(-1, max_idx, False)
        mask.scatter_(-1, min_idx, False)
        vals = vals[mask].view(*vals.shape[:-1], vals.shape[-1]-2)
    
    median_vals = vals[..., 0]
    
    if dim != x.ndim - 1:
        median_vals = median_vals.movedim(-1, dim)
    
    return median_vals

# --- Benchmark ---
start = time.time()
y_bitonic = median_bitonic_step1(x, window_length)
t_bitonic = time.time() - start

start = time.time()
y_elim = median_elimination_step1(x, window_length)
t_elim = time.time() - start

y_diff = torch.allclose(y_bitonic, y_elim)

t_bitonic, t_elim, y_diff


(0.15637874603271484, 0.39742326736450195, False)

In [15]:
# Real median
windows = x.unfold(-1, window_length, 1)
real_median = windows.median(-1).values
bit_diff = torch.allclose(y_bitonic, real_median)
elim_diff = torch.allclose(y_elim, real_median)

bit_diff, elim_diff

(False, True)

In [17]:
import torch

# --- Funzione eliminazione circolare (unificata) ---
def median_elimination_circular(x: torch.Tensor, window_length: int, dim: int = 1) -> torch.Tensor:
    dim = dim % x.ndim
    L = x.shape[dim]

    if window_length > 1:
        idx = [slice(None)] * x.ndim
        idx[dim] = slice(0, window_length-1)
        circular_pad = x[tuple(idx)]
        x_padded = torch.cat([x, circular_pad], dim=dim)
    else:
        x_padded = x

    if dim != x.ndim - 1:
        x_padded = x_padded.movedim(dim, -1)

    windows = x_padded.unfold(-1, window_length, 1)
    vals = windows.clone()
    n = window_length

    for _ in range(n // 2):
        max_idx = vals.argmax(dim=-1, keepdim=True)
        min_idx = vals.argmin(dim=-1, keepdim=True)
        mask = torch.ones_like(vals, dtype=torch.bool)
        mask.scatter_(-1, max_idx, False)
        mask.scatter_(-1, min_idx, False)
        vals = vals[mask].view(*vals.shape[:-1], vals.shape[-1]-2)

    median_vals = vals[..., 0]

    if dim != x.ndim - 1:
        median_vals = median_vals.movedim(-1, dim)

    return median_vals

# --- Esempio vettore 2D ---
x4d = torch.randn(2, 6, 4, 5)  # [N, C, H, W]

window_length = 3

# --- Mediana con eliminazione circolare ---
y_elim = median_elimination_circular(x4d, window_length, dim=1)

# --- Mediana con torch.median su finestre unfold ---
# Creiamo le finestre con padding circolare
pad = window_length - 1
x_padded = torch.cat([x4d, x4d[:, :pad]], dim=1)
windows = x_padded.unfold(1, window_length, 1)  # [B, n_windows=L, window_length]

# mediana usando torch.median
y_torch = windows.median(dim=-1).values  # mediana lungo ultima dimensione

# --- Confronto ---
print("Output eliminazione circolare:\n", y_elim)
print("Output torch.median:\n", y_torch)
print("Differenze:", (y_elim - y_torch))
print("Sono uguali?", torch.allclose(y_elim, y_torch))


Output eliminazione circolare:
 tensor([[[[ 1.1142,  1.1068, -0.3656,  0.2598,  0.5935],
          [-0.1536,  0.0183, -0.2280, -0.4796, -0.1087],
          [ 0.6764, -0.1543, -0.2335, -0.2871,  0.1706],
          [-0.8152,  0.1463,  0.5967, -0.2795, -0.1878]],

         [[ 0.0975,  0.2054,  0.6511,  0.1706,  0.5973],
          [-0.1536,  0.0183, -1.9520, -1.1030, -0.5068],
          [-0.1179, -0.1543, -1.5147, -0.2871,  0.1706],
          [ 0.0760,  0.1463,  1.2234,  0.2242,  0.8022]],

         [[ 0.3227, -0.5270,  0.6511, -0.0751,  0.5973],
          [-0.6356, -0.1007, -1.9520, -1.1030, -0.9925],
          [-0.1179, -0.1543,  0.1262,  1.4428,  0.1706],
          [-0.7515,  0.1463,  1.3844,  0.3947, -0.1878]],

         [[ 0.3227, -0.5270, -0.3211,  0.1706,  0.0059],
          [ 0.1978, -0.5280, -1.5684, -0.5540, -1.1116],
          [ 0.1056,  0.0750, -0.2860,  1.4428, -0.8486],
          [-0.7515, -0.4930,  1.2234,  0.9567, -0.9850]],

         [[ 0.4057, -0.5000, -0.6830,  0.2598,  

In [3]:
import torch
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Usando dispositivo: {device}")

# Funzione originale senza padding
def median_elimination_circular_no_pad(x: torch.Tensor, window_length: int, dim: int = 1) -> torch.Tensor:
    dim = dim % x.ndim

    if dim != x.ndim - 1:
        x = x.movedim(dim, -1)

    windows = x.unfold(-1, window_length, 1)
    vals = windows.clone()
    n = window_length

    for _ in range(n // 2):
        max_idx = vals.argmax(dim=-1, keepdim=True)
        min_idx = vals.argmin(dim=-1, keepdim=True)
        mask = torch.ones_like(vals, dtype=torch.bool)
        mask.scatter_(-1, max_idx, False)
        mask.scatter_(-1, min_idx, False)
        vals = vals[mask].view(*vals.shape[:-1], vals.shape[-1]-2)

    median_vals = vals[..., 0]

    if dim != x.ndim - 1:
        median_vals = median_vals.movedim(-1, dim)

    return median_vals

# Funzione vettorializzata senza padding
def median_elimination_circular_vectorized_no_pad(x: torch.Tensor, window_length: int, dim: int = 1) -> torch.Tensor:
    dim = dim % x.ndim

    if dim != x.ndim - 1:
        x = x.movedim(dim, -1)

    windows = x.unfold(-1, window_length, 1)
    sorted_windows, _ = windows.sort(dim=-1)
    median_idx = window_length // 2
    median_vals = sorted_windows[..., median_idx]

    if dim != x.ndim - 1:
        median_vals = median_vals.movedim(-1, dim)

    return median_vals

# Tensore di test
torch.manual_seed(0)
x = torch.randn(1000, 50, device=device)
window_length = 5
dim = 1

# Funzione originale
start = time.time()
med_orig = median_elimination_circular(x, window_length, dim)
time_orig = time.time() - start

# Funzione vettorializzata
start = time.time()
med_vect = median_elimination_circular_vectorized(x, window_length, dim)
time_vect = time.time() - start

# Torch.median con padding circolare
if window_length > 1:
    idx = [slice(None)] * x.ndim
    idx[dim] = slice(0, window_length-1)
    circular_pad = x[tuple(idx)]
    x_padded = torch.cat([x, circular_pad], dim=dim)
else:
    x_padded = x

windows = x_padded.unfold(dim, window_length, 1)
med_torch, _ = windows.median(dim=-1)

start = time.time()
windows = x.unfold(dim, window_length, 1)
med_torch, _ = windows.median(dim=-1)
time_med = time.time() - start

# Verifica correttezza
correct_orig = torch.allclose(med_orig, med_torch)
correct_vect = torch.allclose(med_vect, med_torch)

print(f"Tempo funzione originale: {time_orig:.6f} s")
print(f"Tempo funzione vettorializzata: {time_vect:.6f} s")
print(f"Tempo funzione tensor.median: {time_med:.6f} s")
print(f"Differenza massimo rispetto a torch.median - originale: {correct_orig}")
print(f"Differenza massimo rispetto a torch.median - vettorializzata: {correct_vect}")


Usando dispositivo: cpu


RuntimeError: The size of tensor a (50) must match the size of tensor b (46) at non-singleton dimension 1

In [5]:
import torch

class Test:
    def __init__(self, window_size):
        self.window_size = window_size

    def median_elimination_circular(self, x: torch.Tensor, dim: int = 1) -> torch.Tensor:
        dim = dim % x.ndim

        if self.window_size > 1:
            idx = [slice(None)] * x.ndim
            idx[dim] = slice(0, self.window_size-1)
            circular_pad = x[tuple(idx)]
            x_padded = torch.cat([x, circular_pad], dim=dim)
        else:
            x_padded = x

        if dim != x.ndim - 1:
            x_padded = x_padded.movedim(dim, -1)

        windows = x_padded.unfold(-1, self.window_size, 1)
        median_vals = windows.median(dim=-1, keepdim=True)[0]

        if dim != x.ndim - 1:
            median_vals = median_vals.movedim(-1, dim)

        return median_vals

# Tensore di test piccolo
x = torch.tensor([[1, 3, 2, 4, 5],
                  [10, 20, 30, 40, 50]], dtype=torch.float32)

window_size = 3
test_obj = Test(window_size)

# Calcolo della mediana su finestre mobili
med_local = test_obj.median_elimination_circular(x, dim=1)

print("Tensore originale:\n", x)
print(f"Mediana su finestre mobili di size={window_size}:\n", med_local)


Tensore originale:
 tensor([[ 1.,  3.,  2.,  4.,  5.],
        [10., 20., 30., 40., 50.]])
Mediana su finestre mobili di size=3:
 tensor([[[ 2.],
         [ 3.],
         [ 4.],
         [ 4.],
         [ 3.]],

        [[20.],
         [30.],
         [40.],
         [40.],
         [20.]]])
